# 01 – Data Quality Assessment

## Objective

This notebook performs a structured data quality assessment of the raw credit application dataset prior to downstream analytical use.

The evaluation focuses on the following quality dimensions:

- **Completeness** – identification and quantification of missing values  
- **Consistency** – detection of inconsistent formats, categorical encodings, and schema variations  
- **Validity** – identification of impossible or out-of-range values  
- **Uniqueness** – detection of duplicate records and conflicting identifiers  

For each identified issue, we quantify its extent, assess its potential analytical impact, and apply justified remediation strategies where appropriate.

The outcome of this process is a cleaned and standardized dataset suitable for reliable fairness analysis and governance evaluation.

In [1]:
# Standard libraries
import json
import copy
from pathlib import Path

# Data manipulation
import pandas as pd
import numpy as np

# Display settings (ensuring all columns are visible)
pd.set_option("display.max_columns", None)

# Define data path
DATA_PATH = Path("../data/raw_credit_applications.json")

In [2]:
# Load raw JSON file
with open(DATA_PATH, "r") as f:
    records = json.load(f)

print(f"Total applications loaded: {len(records)}")

Total applications loaded: 502


In [3]:
# Inspect structure of first record
print("Top-level fields in first application record:")
print(records[0].keys())

Top-level fields in first application record:
dict_keys(['_id', 'applicant_info', 'financials', 'spending_behavior', 'decision', 'processing_timestamp'])


In [4]:
# Validate structural consistency across all records

# Collect the set of top-level keys for each record
key_sets = [set(r.keys()) for r in records]

# Identify unique structural patterns
unique_structures = {tuple(sorted(keys)) for keys in key_sets}

print(f"Number of distinct top-level structures: {len(unique_structures)}")

Number of distinct top-level structures: 4


In [5]:
# Display distinct top-level structures

for i, structure in enumerate(unique_structures, 1):
    print(f"\nStructure {i}:")
    print(structure)


Structure 1:
('_id', 'applicant_info', 'decision', 'financials', 'loan_purpose', 'spending_behavior')

Structure 2:
('_id', 'applicant_info', 'decision', 'financials', 'spending_behavior')

Structure 3:
('_id', 'applicant_info', 'decision', 'financials', 'notes', 'spending_behavior')

Structure 4:
('_id', 'applicant_info', 'decision', 'financials', 'processing_timestamp', 'spending_behavior')


### Top-Level Schema Variation

The dataset exhibits four distinct top-level schema variations.

A core schema structure is consistently present across all records, comprising:
- `_id`
- `applicant_info`
- `financials`
- `decision`
- `spending_behavior`

However, the following additional fields appear inconsistently:
- `loan_purpose`
- `processing_timestamp`
- `notes`

This indicates that certain attributes are optional and inconsistently populated across the dataset. Prior to transformation and analysis, the schema must be unified to ensure consistent representation of all expected fields.

In [6]:
# Quantify presence of optional top-level fields

optional_fields = ["loan_purpose", "processing_timestamp", "notes"]

total_records = len(records)

summary = []

for field in optional_fields:
    present = sum(field in r for r in records)
    summary.append({
        "field": field,
        "present_count": present,
        "present_pct (%)": round((present / total_records) * 100, 2)
    })

pd.DataFrame(summary).sort_values("present_count", ascending=False).reset_index(drop=True)

,field,present_count,present_pct (%)
0,processing_timestamp,62,12.35
1,loan_purpose,50,9.96
2,notes,2,0.40


### Schema Normalization

The dataset contains a consistent core structure, but several top-level fields appear only in subsets of records (e.g., `loan_purpose`, `processing_timestamp`, `notes`). 

To ensure consistent processing and reproducible transformation steps, we standardize the top-level schema by explicitly adding these optional fields to all records when absent and representing missing values as null. This does not imply that these fields are mandatory. It enforces a uniform schema representation prior to flattening and downstream analysis.

In [7]:
records_norm = copy.deepcopy(records)  # work on a copy; keep `records` as raw reference

In [8]:
# Define a canonical top-level schema for consistent processing

canonical_fields = [
    "_id",
    "applicant_info",
    "financials",
    "decision",
    "spending_behavior",
    "loan_purpose",
    "processing_timestamp",
    "notes",
]

In [9]:
# Normalize records_norm: ensure all canonical fields exist in every record
# Missing optional fields are added as explicit nulls (None)

for r in records_norm:
    for field in canonical_fields:
        if field not in r:
            r[field] = None

In [10]:
# Verify that all records now share the same top-level structure after normalization
normalized_structures = {tuple(sorted(r.keys())) for r in records_norm}

assert len(normalized_structures) == 1, "Schema normalization failed"
print(f"All {len(records_norm)} records now share a unified top-level schema.")

All 502 records now share a unified top-level schema.


In [11]:
df_discovery = pd.json_normalize(records_norm, sep=".")
df_discovery.columns = [c.replace(".", "_") for c in df_discovery.columns]

print("df_discovery shape:", df_discovery.shape)
df_discovery.head()

df_discovery shape: (502, 21)


,_id,spending_behavior,processing_timestamp,loan_purpose,notes,applicant_info_full_name,applicant_info_email,applicant_info_ssn,applicant_info_ip_address,applicant_info_gender,applicant_info_date_of_birth,applicant_info_zip_code,financials_annual_income,financials_credit_history_months,financials_debt_to_income,financials_savings_balance,decision_loan_approved,decision_rejection_reason,decision_interest_rate,decision_approved_amount,financials_annual_salary
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,None,None,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",None,None,None,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,78000,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",None,vacation,None,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000,41,0.21,37909,True,NaN,3.7,59000.0,NaN
3,app_024,"[{'category': 'Fitness', 'amount': 575}]",None,None,None,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,103000,70,0.35,0,True,NaN,4.3,34000.0,NaN
4,app_184,"[{'category': 'Entertainment', 'amount': 463}]",2024-01-15T00:00:00Z,None,None,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,M,1999-05-21,10080,57000,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN


### Schema drift check (core fields)

Before cleaning, we check whether the same concept appears under different field names (e.g., income recorded as `annual_income` vs `annual_salary`). If so, we will standardize it to avoid treating valid values as missing.

In [12]:
# Schema overview: how many columns per nested section?
app_cols = [c for c in df_discovery.columns if c.startswith("applicant_info_")]
fin_cols = [c for c in df_discovery.columns if c.startswith("financials_")]
dec_cols = [c for c in df_discovery.columns if c.startswith("decision_")]

print(f"Applicant fields: {len(app_cols)}")
print(f"Financial fields: {len(fin_cols)}")
print(f"Decision fields: {len(dec_cols)}")

# Focused drift check: income fields (annual_income vs annual_salary)
income_like = [c for c in fin_cols if ("annual_income" in c) or ("annual_salary" in c)]
print("\nIncome-related financial fields detected:", income_like)

# Quantify presence (non-null) to show drift impact
for col in income_like:
    non_null = df_discovery[col].notna().sum()
    print(f"  {col}: non-null in {non_null} records ({(non_null/len(df_discovery))*100:.2f}%)")

Applicant fields: 7
Financial fields: 5
Decision fields: 4

Income-related financial fields detected: ['financials_annual_income', 'financials_annual_salary']
  financials_annual_income: non-null in 497 records (99.00%)
  financials_annual_salary: non-null in 5 records (1.00%)


Income is recorded under two alternative fields. `financials_annual_income` is populated for 497/502 applications, while `financials_annual_salary` is populated for 5/502. These represent the same concept and will be merged into a single canonical `financials_annual_income_canonical` field in the curated dataset.

In [13]:
# Structural fix: merge annual_income and annual_salary into one canonical field

df_work = df_discovery.copy()

income_col = "financials_annual_income"
salary_col = "financials_annual_salary"

income_num = pd.to_numeric(df_work[income_col], errors="coerce")
salary_num = pd.to_numeric(df_work[salary_col], errors="coerce")

df_work["financials_annual_income_canonical"] = income_num.combine_first(salary_num)

# Structural verification
missing_after_merge = df_work["financials_annual_income_canonical"].isna().sum()

print(
    "Missing canonical income after structural merge:",
    missing_after_merge,
    "/",
    len(df_work)
)

Missing canonical income after structural merge: 0 / 502


In [24]:
# Exclude two columns from DQ analysis - replaced by canonical income)
exclude_cols = [
    "financials_annual_income",
    "financials_annual_salary"
]

## 2. Completeness

After resolving structural inconsistencies,  we assess missing values across key column groups. The objective is to quantify incomplete records before applying any remediation.

In [29]:
# Column grouping by schema section
core_cols      = [c for c in df_work.columns if c in ["_id", "processing_timestamp", "loan_purpose", "notes"]]
applicant_info_cols = [c for c in df_work.columns if c.startswith("applicant_info_")]
financial_cols = [
    c for c in df_work.columns
    if c.startswith("financials_")
    and c not in exclude_cols
]
spending_cols  = [c for c in df_work.columns if c.startswith("spending_")]
decision_cols  = [c for c in df_work.columns if c.startswith("decision_")]

column_groups = {
    "core":      core_cols,
    "applicant": applicant_info_cols,
    "financial": financial_cols,
    "spending":  spending_cols,
    "decision":  decision_cols,
}

# Fields where missingness is structurally expected:
# - Decision fields: rejected apps have no interest rate or approved amount;
#                   approved apps have no rejection reason
# - Optional fields: documented during schema inspection as inconsistently
#                   populated by design, not data quality failures
EXPECTED_MISSING = {
    "decision_interest_rate",
    "decision_approved_amount",
    "decision_rejection_reason",
    "notes",                     # optional — present in 0.40% of records
    "loan_purpose",              # optional — present in 9.96% of records
    "processing_timestamp",      # optional — present in 12.35% of records
}

# Completeness calculation
n_records = len(df_work)
rows = []

for group_name, cols in column_groups.items():
    for col in cols:
        missing_count = df_work[col].isna().sum()
        rows.append({
            "column":        col,
            "group":         group_name,
            "missing_count": missing_count,
            "missing_pct":   round(missing_count / n_records * 100, 2),
            "expected":      col in EXPECTED_MISSING,
        })

completeness_df = (
    pd.DataFrame(rows)
    .sort_values(["missing_count", "group"], ascending=[False, True])
    .reset_index(drop=True)
)

completeness_df

,column,group,missing_count,missing_pct,expected
0,notes,core,500,99.60,True
1,loan_purpose,core,452,90.04,True
2,processing_timestamp,core,440,87.65,True
3,decision_rejection_reason,decision,292,58.17,True
4,decision_interest_rate,decision,210,41.83,True
5,decision_approved_amount,decision,210,41.83,True
6,applicant_info_ssn,applicant,5,1.00,False
7,applicant_info_ip_address,applicant,5,1.00,False
8,applicant_info_gender,applicant,1,0.20,False
9,applicant_info_date_of_birth,applicant,1,0.20,False


High missingness is mainly concentrated in optional fields such as `notes`, `loan_purpose`, and `processing_timestamp`, so it is treated as expected.

In [30]:
# Identify columns with unexpected missing values
genuine_issues = completeness_df[
    (completeness_df["missing_count"] > 0) & 
    (completeness_df["expected"] == False)
]

print(f"Columns with unexpected missing values: {len(genuine_issues)}")
genuine_issues

Columns with unexpected missing values: 5


,column,group,missing_count,missing_pct,expected
6,applicant_info_ssn,applicant,5,1.0,False
7,applicant_info_ip_address,applicant,5,1.0,False
8,applicant_info_gender,applicant,1,0.2,False
9,applicant_info_date_of_birth,applicant,1,0.2,False
10,applicant_info_zip_code,applicant,1,0.2,False


The table above shows missing values in key applicant identifiers and demographic fields (SSN, IP address, gender, date of birth, and ZIP code). Even though the counts are low, these attributes are important for traceability and governance. The affected records will be flagged and handled during remediation to ensure the curated dataset remains reliable.